In [20]:
import os
import sys
import yaml
import pandas as pd
import json

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import utils

In [21]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [22]:
# for now, for replicability, limit to 2024-12-19 as last date
#meetings_df = meetings_df[:157]
#meetings_df.tail(1)

In [23]:
# agenda items dataframe

agenda_items_df = []
for _, row in meetings_df.iterrows():
    date = row['date']
    year = date[0:4]
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized-2.json")
    with open(input_filepath, 'r', encoding='utf-8') as f:
        agenda_items = json.load(f)
    for item in agenda_items:
        my_row = {
            "date": date,
            "year": year,
            "item_no": item['item_no'],
            "sub_item_no": item['sub_item_no'],
            "is_consent_calendar": item['is_consent_calendar'],
            "agenda_text": item['text']
        }
        for col in ['title', 'address', 'council_district', 'council_member', 'last_day_to_act', 'referenced_laws', 'is_appeal', 'summary_of_appeal', 'summary']:
            if col=='referenced_laws':
                my_row[col] = item['ai_summary'].get(col, [])
            else:
                my_row[col] = item['ai_summary'].get(col, "")
        for col in ['project_type', 'new_or_existing', 'square_footage', 'dwelling_units', 'affordable_units', 'height', 'parking_spaces', 'requested_actions']:
            if col=='requested_actions':
                my_row[col] = item['ai_summary_2'].get(col, [])
            else:
                my_row[col] = item['ai_summary_2'].get(col, "")
        my_row['agenda_summary'] = my_row.pop('summary')
        my_row['is_casenum'] = utils.is_casenum(my_row['title'])
        agenda_items_df.append(my_row)

agenda_items_df = pd.DataFrame(agenda_items_df)
agenda_items_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "agenda_items.parquet"), index=False)
    

In [24]:
# minutes items dataframe

minutes_items_df = []
for _, row in meetings_df.iterrows():
    date = row['date']
    year = date[0:4]
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized-minutes.json")
    with open(input_filepath, 'r', encoding='utf-8') as f:
        minutes_items = json.load(f)
    for item in minutes_items:
        if not item['ai_minutes_summary']:
            continue
        my_row = {
            "date": date,
            "year": year,
            "item_no": item['item_no'],
            "sub_item_no": item['sub_item_no'],
        }
        for col in ['summary', 'item_order_changed', 'taken_off_consent_calendar', 'multiple_votes', 'mover', 'seconder', 'ayes', 'noes', 'abstentions', 'absences', 'recusals', 'vote_result', 'appeal_result', 'project_result']:
            if col in ['ayes', 'noes', 'abstentions', 'absences', 'recusals']:
                my_row[col] = item['ai_minutes_summary'].get(col, [])
            else:
                my_row[col] = item['ai_minutes_summary'].get(col, "")
        my_row['minutes_summary'] = my_row.pop('summary')
        minutes_items_df.append(my_row)

minutes_items_df = pd.DataFrame(minutes_items_df)
minutes_items_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "minutes_items.parquet"), index=False)   


In [25]:
# supplemental docs dataframe (+expanded by referenced item)

supplemental_docs_df = []
docs_by_item_df = []

for _, row in meetings_df.iterrows():
    date = row['date']
    year = date[0:4]
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "supplemental-docs-summarized-2.json")
    docs_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "supplemental-docs.pkl")
    docs_df = pd.read_pickle(docs_filepath)
    with open(input_filepath, 'r', encoding='utf-8') as f:
        supplemental_docs = json.load(f)
    for doc in supplemental_docs:
        my_row = {
            "date": date,
            "year": year,
            "doc_id": doc["doc_id"],
            "text": docs_df.loc[(docs_df["date"]==date) & (docs_df["doc_id"]==doc["doc_id"]), "content"].values[0]
        }
        for col in ['summary', 'referenced_agenda_items', 'document_type', 'author_type', 'support_or_oppose', 'support_or_oppose_reasons', 'support_or_oppose_explanation']:
            my_row[col] = doc[col]
        supplemental_docs_df.append(my_row)

        referenced_agenda_items = doc['referenced_agenda_items']
        for item in referenced_agenda_items:
            support_or_oppose = doc['support_or_oppose'].get(item, "")
            support_or_oppose_reasons = doc['support_or_oppose_reasons'].get(item, [])
            support_or_oppose_explanation = doc['support_or_oppose_explanation'].get(item, "")
            my_row = {
                "date": date,
                "year": year,
                "doc_id": doc["doc_id"],
                "item_no": item,
                "support_or_oppose": support_or_oppose,
                "support_or_oppose_reasons": support_or_oppose_reasons,
                "support_or_oppose_explanation": support_or_oppose_explanation
            }
            docs_by_item_df.append(my_row)


supplemental_docs_df = pd.DataFrame(supplemental_docs_df)
supplemental_docs_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "supplemental_docs.parquet"), index=False)

docs_by_item_df = pd.DataFrame(docs_by_item_df)
docs_by_item_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "docs_by_item.parquet"), index=False)



In [26]:
# Main analysis dataframe

df = minutes_items_df.merge(
    agenda_items_df, 
    on=['date', 'year', 'item_no', 'sub_item_no'],
    how='inner'
).reset_index(drop=True)

# some cleaning

df['is_appeal'] = (df['is_appeal']=="yes")
df['item_order_changed'] = (df['item_order_changed']=="YES")
df['taken_off_consent_calendar'] = (df['taken_off_consent_calendar']=="YES")
df['multiple_votes'] = (df['multiple_votes']=="YES")

# name standardization

with open(os.path.join(LOCAL_PATH, "intermediate_data/cpc/names-standardized.json"), 'r', encoding='utf-8') as f:
    names_mapping = json.load(f)

df['mover'] = df['mover'].map(names_mapping)
df['seconder'] = df['seconder'].map(names_mapping)
df['ayes'] = df['ayes'].apply(lambda x: [names_mapping[name] for name in x])
df['noes'] = df['noes'].apply(lambda x: [names_mapping[name] for name in x])
df['abstentions'] = df['abstentions'].apply(lambda x: [names_mapping[name] for name in x])
df['absences'] = df['absences'].apply(lambda x: [names_mapping[name] for name in x])
df['recusals'] = df['recusals'].apply(lambda x: [names_mapping[name] for name in x])

# num ayes and noes
df['num_ayes'] = df['ayes'].apply(len)
df['num_noes'] = df['noes'].apply(len)
df['unanimous'] = (df['num_ayes']>0) & (df['num_noes']==0)

# count docs
df['n_docs'] = 0
df['n_support'] = 0
df['n_oppose'] = 0

for idx, row in df.iterrows():
    date = row['date']
    year = row['year']
    item_no = row['item_no']
    mask = (docs_by_item_df['date']==date) & (docs_by_item_df['year']==year) & (docs_by_item_df['item_no']==item_no)
    df.at[idx, 'n_docs'] = mask.sum()
    df.at[idx, 'n_support'] = docs_by_item_df.loc[mask, 'support_or_oppose'].isin(["DEFINITELY SUPPORT", "SOMEWHAT SUPPORT"]).sum()
    df.at[idx, 'n_oppose'] = docs_by_item_df.loc[mask, 'support_or_oppose'].isin(["DEFINITELY OPPOSE", "SOMEWHAT OPPOSE"]).sum()

df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data.parquet"), index=False)


In [27]:
# Reasons dataset

REASON_MAP = {
    "NEIGHBORHOOD CHARACTER": "neighborhood character",
    "PROCEDURAL MATTERS": "procedural issues",
    "BUILDING HEIGHT / SIZE / DENSITY": "building height / size / density",
    "HOUSING AFFORDABILITY / AFFORDABLE HOUSING": "housing supply / affordability",
    "CLIMATE / ENVIRONMENT": "climate / environment",
    "SAFETY": "safety",
    "CEQA RELATED MATTERS": "CEQA related matters",
    "COMMUNITY ENGAGEMENT / OPPOSITION": "community engagement",
    "HOUSING SUPPLY": "housing supply / affordability",
    "DISPLACEMENT / GENTRIFICATION": "displacement / gentrification",
    "TRAFFIC IMPACT": "traffic / parking",
    "TRANSIT ACCESS": "transit access / walkability",
    "ECONOMIC IMPACTS / JOBS": "economic impacts / jobs",
    "PARKING IMPACT": "traffic / parking",
    "HISTORIC / CULTURAL PRESERVATION": "historic / cultural preservation",
    "HOMELESSNESS": "homelessness",
    "FAIR HOUSING / HOUSING EQUITY": "housing supply / affordability",
    "NOISE / LIGHT POLLUTION": "neighborhood character",
    "WALKABILITY / BIKEABILITY": "transit access / walkability",
    "UNION LABOR / PREVAILING WAGES": "procedural issues"
}

reasons_df = []
for _, row in docs_by_item_df.iterrows():
    date = row['date']
    year = row['year']
    doc_id = row['doc_id']
    item_no = row['item_no']
    support_or_oppose = row['support_or_oppose']
    for r in row['support_or_oppose_reasons']:
        reason = REASON_MAP[r]
        reasons_df.append({
            'date': date,
            'year': year,
            'doc_id': doc_id,
            'item_no': item_no,
            'support_or_oppose': support_or_oppose,
            'reason': reason
        })

reasons_df = pd.DataFrame(reasons_df)

reasons_df = reasons_df.drop_duplicates()

reasons_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "reasons.parquet"), index=False)


In [28]:
df.columns

Index(['date', 'year', 'item_no', 'sub_item_no', 'item_order_changed',
       'taken_off_consent_calendar', 'multiple_votes', 'mover', 'seconder',
       'ayes', 'noes', 'abstentions', 'absences', 'recusals', 'vote_result',
       'appeal_result', 'project_result', 'minutes_summary',
       'is_consent_calendar', 'agenda_text', 'title', 'address',
       'council_district', 'council_member', 'last_day_to_act',
       'referenced_laws', 'is_appeal', 'summary_of_appeal', 'project_type',
       'new_or_existing', 'square_footage', 'dwelling_units',
       'affordable_units', 'height', 'parking_spaces', 'requested_actions',
       'agenda_summary', 'is_casenum', 'num_ayes', 'num_noes', 'unanimous',
       'n_docs', 'n_support', 'n_oppose'],
      dtype='str')